In [1]:
import hydra
from omegaconf import OmegaConf
import site_archive_nci

config_path = '/g/data/kd24/tjl/src/PyEarthTools/packages/bundled_models/fourcastnext/Training/limited_variables_early_stopping'


In [2]:
initialsed = hydra.initialize_config_dir(version_base=None, config_dir=config_path)
cfg = hydra.compose(config_name="limited_vars_early_stop.yaml")

In [3]:
import fourcastnext
model = fourcastnext.registered_model.FourCastNextRM(
    pipeline='early_stopping',
    output='.',
    ckpt_path='/scratch/kd24/ML/model-epoch=00-step=5000.ckpt',
    lead_time=24
)

/home/548/tjl548/.local/lib/python3.11/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [4]:
model

`pyearthtools.zoo` Forecast model

Model Name:          'Development/FourCastNextRM'
Data Source:         'early_stopping'
Output Directory:    '.'



In [5]:
wrapper, kwags = model.load()

/g/data/kd24/tjl/src/PyEarthTools/packages/training/src/pyearthtools/training/wrapper/lightning/wrapper.py:121: Loading checkpoint: /scratch/kd24/ML/model-epoch=00-step=5000.ckpt


In [13]:
import torch
torch.save(wrapper.model.model, "/scratch/kd24/ML/test.torch")

In [17]:
import torch
backend = "qnnpack"
wrapper.model.model.qconfig = torch.quantization.get_default_qconfig(backend)
torch.backends.quantized.engine = backend
model_static_quantized = torch.quantization.prepare(wrapper.model.model.to('cpu'), inplace=False)
model_static_quantized = torch.quantization.convert(model_static_quantized, inplace=False)
torch.save(model_static_quantized, "/scratch/kd24/ML/quantized.torch")

/opt/conda/envs/pet/lib/python3.11/site-packages/torch/ao/quantization/observer.py:1318: UserWarning: must run observer before calling calculate_qparams.                                    Returning default scale and zero point 
  warnings.warn(
